# Modelagem V2

Este notebook constrói a segunda versão da base analítica e repete a etapa de Machine Learning após a inclusão de indicadores históricos do IDEB.

Os objetivos desta etapa são:

- incorporar as novas features educacionais à base analítica;
- manter a mesma estratégia de split utilizada na primeira rodada;
- repetir os mesmos modelos e hiperparâmetros;
- comparar diretamente o desempenho da V1 e da V2.

O conjunto de teste temporal de 2024 permanecerá reservado durante o desenvolvimento.

# 1. Configuração

Nesta etapa são importadas as bibliotecas utilizadas no notebook.

In [0]:
from pyspark.sql import functions as F

from pyspark.ml import Pipeline

from pyspark.ml.feature import (
    Imputer,
    StringIndexer,
    OneHotEncoder,
    VectorAssembler,
    StandardScaler,
)

from pyspark.ml.classification import (
    LogisticRegression,
    DecisionTreeClassifier,
    RandomForestClassifier,
)

from pyspark.ml.evaluation import BinaryClassificationEvaluator

# 2. Construção da Base V2

A base analítica original é enriquecida com as duas features selecionadas do IDEB 2021.

A associação é realizada por município e rede.

## Carregamento

In [0]:
df_gold_v1 = spark.table("workspace.gold.ml_alfabetizacao_aluno")

df_ideb = spark.table("workspace.gold.ml_features_ideb_2021")

## Join

In [0]:
df_gold_v2 = (
    df_gold_v1.alias("base")
    .join(
        df_ideb.alias("ideb"),
        (F.col("base.id_municipio") == F.col("ideb.id_municipio"))
        & (F.col("base.rede") == F.col("ideb.rede_codigo")),
        how="left",
    )
    .select(
        "base.*",
        F.col("ideb.ideb_saeb_portugues_2021").alias("ideb_saeb_portugues_2021"),
        F.col("ideb.ideb_indicador_rendimento_2021").alias(
            "ideb_indicador_rendimento_2021"
        ),
    )
)

Validação

In [0]:
display(
    df_gold_v2.agg(
        F.count("*").alias("qtd_alunos"),
        F.count("ideb_saeb_portugues_2021").alias("com_saeb_portugues"),
        F.count("ideb_indicador_rendimento_2021").alias("com_indicador_rendimento"),
    )
)

In [0]:
print("V1:", df_gold_v1.count())

print("V2:", df_gold_v2.count())

## Persistência da Base V2

A nova versão da base é persistida separadamente para preservar a primeira versão e permitir comparações reproduzíveis.

In [0]:
(
    df_gold_v2.write.format("delta")
    .mode("overwrite")
    .partitionBy("ano")
    .saveAsTable("workspace.gold.ml_alfabetizacao_aluno_v2")
)

# 3. Conjuntos de Modelagem

É reutilizada a mesma atribuição de treino, validação e teste da primeira rodada.

Dessa forma, qualquer diferença de desempenho entre V1 e V2 pode ser atribuída às novas features, e não a mudanças na amostra.

## Carregando bases

In [0]:
df_gold = spark.table("workspace.gold.ml_alfabetizacao_aluno_v2")
df_split = spark.table("workspace.gold.ml_split_modelagem")

In [0]:
df_modelagem = df_gold.join(df_split, on=["ano", "id_municipio"], how="inner")

df_treino = df_modelagem.filter(F.col("split") == "treino")

df_validacao = df_modelagem.filter(F.col("split") == "validacao")

df_teste = df_modelagem.filter(F.col("split") == "teste")

In [0]:
display(
    df_modelagem.groupBy("split")
    .agg(
        F.count("*").alias("qtd_alunos"),
        F.countDistinct("id_municipio").alias("qtd_municipios"),
        F.round(
            F.avg(F.col("alfabetizado").cast("double")) * 100,
            2,
        ).alias("taxa_alfabetizacao"),
    )
    .orderBy("split")
)

# 4. Definição das Features

A V2 mantém todas as features da primeira rodada e adiciona duas variáveis históricas derivadas do IDEB 2021.

As novas variáveis representam desempenho em Língua Portuguesa e fluxo escolar.

In [0]:
target = "alfabetizado"

features_categoricas = ["rede"]

features_numericas = [
    "inse_2021",
    "censo_pct_agua_potavel",
    "censo_pct_energia_rede_publica",
    "censo_pct_esgoto_rede_publica",
    "censo_pct_biblioteca_sala_leitura",
    "censo_pct_laboratorio_informatica",
    "censo_pct_internet_aprendizagem",
    "censo_docentes_por_100_alunos",
    "censo_turmas_por_100_alunos",
    "log_pib_per_capita",
    "log_populacao",
    "ideb_saeb_portugues_2021",
    "ideb_indicador_rendimento_2021",
]

features_modelo = features_categoricas + features_numericas

# 5. Baseline Ingênuo

O mesmo baseline da primeira rodada é mantido como referência.

Como a classe majoritária corresponde aos alunos alfabetizados, o classificador ingênuo prevê `alfabetizado = 1` para todos os registros.

In [0]:
df_baseline_validacao = df_validacao.withColumn("prediction", F.lit(1.0))

In [0]:
display(
    df_baseline_validacao.groupBy("alfabetizado", "prediction")
    .count()
    .orderBy("alfabetizado", "prediction")
)

In [0]:
baseline_metricas = df_baseline_validacao.agg(
    F.count("*").alias("total"),
    F.sum(
        F.when(
            F.col("alfabetizado") == F.col("prediction"),
            1,
        ).otherwise(0)
    ).alias("acertos"),
    F.sum(
        F.when(
            (F.col("alfabetizado") == 0) & (F.col("prediction") == 0),
            1,
        ).otherwise(0)
    ).alias("acertos_classe_0"),
    F.sum(
        F.when(
            F.col("alfabetizado") == 0,
            1,
        ).otherwise(0)
    ).alias("total_classe_0"),
)

In [0]:
display(
    baseline_metricas.select(
        F.round(
            F.col("acertos") / F.col("total"),
            4,
        ).alias("accuracy"),
        F.round(
            F.col("acertos_classe_0") / F.col("total_classe_0"),
            4,
        ).alias("recall_classe_0"),
    )
)

# 6. Pipeline de Preprocessing

O mesmo pipeline da primeira rodada é reutilizado.

A imputação e os demais tratamentos continuam sendo ajustados exclusivamente sobre o conjunto de treino.

In [0]:
df_treino_ml = df_treino.withColumn("label", F.col("alfabetizado").cast("double"))

df_validacao_ml = df_validacao.withColumn("label", F.col("alfabetizado").cast("double"))

In [0]:
features_numericas_imputadas = [f"{coluna}_imp" for coluna in features_numericas]

imputer = Imputer(
    inputCols=features_numericas,
    outputCols=features_numericas_imputadas,
    strategy="median",
)

rede_indexer = StringIndexer(
    inputCol="rede",
    outputCol="rede_index",
    handleInvalid="keep",
)

rede_encoder = OneHotEncoder(
    inputCols=["rede_index"],
    outputCols=["rede_ohe"],
    handleInvalid="keep",
)

assembler = VectorAssembler(
    inputCols=(features_numericas_imputadas + ["rede_ohe"]),
    outputCol="features_raw",
)

scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withStd=True,
    withMean=False,
)

# 7. Logistic Regression

A Logistic Regression é novamente utilizada como primeiro modelo supervisionado.

Os mesmos parâmetros da V1 são mantidos para permitir comparação direta entre as duas versões da base.

## Treino

In [0]:
logistic = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    maxIter=100,
    regParam=0.0,
    elasticNetParam=0.0,
)

pipeline_logistic = Pipeline(
    stages=[
        imputer,
        rede_indexer,
        rede_encoder,
        assembler,
        scaler,
        logistic,
    ]
)

In [0]:
modelo_logistic = pipeline_logistic.fit(df_treino_ml)

pred_logistic_validacao = modelo_logistic.transform(df_validacao_ml)

## Métricas

In [0]:
evaluator_auc = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC",
)

auc_logistic = evaluator_auc.evaluate(pred_logistic_validacao)

print(f"ROC-AUC validação: {auc_logistic:.4f}")

In [0]:
display(
    pred_logistic_validacao.groupBy("label", "prediction")
    .count()
    .orderBy("label", "prediction")
)

In [0]:
def calcular_metricas_classificacao(df_pred):
    resumo = df_pred.agg(
        F.count("*").alias("total"),
        F.sum(
            F.when(
                (F.col("label") == 1) & (F.col("prediction") == 1),
                1,
            ).otherwise(0)
        ).alias("tp"),
        F.sum(
            F.when(
                (F.col("label") == 0) & (F.col("prediction") == 0),
                1,
            ).otherwise(0)
        ).alias("tn"),
        F.sum(
            F.when(
                (F.col("label") == 0) & (F.col("prediction") == 1),
                1,
            ).otherwise(0)
        ).alias("fp"),
        F.sum(
            F.when(
                (F.col("label") == 1) & (F.col("prediction") == 0),
                1,
            ).otherwise(0)
        ).alias("fn"),
    ).collect()[0]

    tp = resumo["tp"]
    tn = resumo["tn"]
    fp = resumo["fp"]
    fn = resumo["fn"]

    accuracy = (tp + tn) / resumo["total"]

    precision_1 = tp / (tp + fp) if (tp + fp) else 0

    recall_1 = tp / (tp + fn) if (tp + fn) else 0

    f1_1 = (
        2 * precision_1 * recall_1 / (precision_1 + recall_1)
        if (precision_1 + recall_1)
        else 0
    )

    precision_0 = tn / (tn + fn) if (tn + fn) else 0

    recall_0 = tn / (tn + fp) if (tn + fp) else 0

    f1_0 = (
        2 * precision_0 * recall_0 / (precision_0 + recall_0)
        if (precision_0 + recall_0)
        else 0
    )

    return {
        "accuracy": accuracy,
        "precision_0": precision_0,
        "recall_0": recall_0,
        "f1_0": f1_0,
        "precision_1": precision_1,
        "recall_1": recall_1,
        "f1_1": f1_1,
    }

In [0]:
metricas_logistic = calcular_metricas_classificacao(pred_logistic_validacao)

for metrica, valor in metricas_logistic.items():
    print(f"{metrica}: {valor:.4f}")

print(f"roc_auc: {auc_logistic:.4f}")

# 8. Decision Tree

A Decision Tree é repetida com os mesmos hiperparâmetros da primeira rodada.

O objetivo é avaliar se as novas features educacionais melhoram a capacidade de separação das classes em relações não lineares.

In [0]:
assembler_tree = VectorAssembler(
    inputCols=(features_numericas_imputadas + ["rede_ohe"]),
    outputCol="features",
)

decision_tree = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    maxDepth=5,
    seed=42,
)

pipeline_tree = Pipeline(
    stages=[
        imputer,
        rede_indexer,
        rede_encoder,
        assembler_tree,
        decision_tree,
    ]
)

modelo_tree = pipeline_tree.fit(df_treino_ml)

pred_tree_validacao = modelo_tree.transform(df_validacao_ml)

auc_tree = evaluator_auc.evaluate(pred_tree_validacao)

In [0]:
print(f"ROC-AUC validação: {auc_tree:.4f}")

display(
    pred_tree_validacao.groupBy("label", "prediction")
    .count()
    .orderBy("label", "prediction")
)

In [0]:
metricas_tree = calcular_metricas_classificacao(pred_tree_validacao)

for metrica, valor in metricas_tree.items():
    print(f"{metrica}: {valor:.4f}")

print(f"roc_auc: {auc_tree:.4f}")

# 9. Random Forest

A Random Forest utiliza os mesmos parâmetros da V1.

Como foi o modelo com maior ROC-AUC na primeira rodada, sua comparação será a principal referência para medir o impacto do enriquecimento.

In [0]:
random_forest = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    numTrees=50,
    maxDepth=8,
    seed=42,
)

In [0]:
pipeline_rf = Pipeline(
    stages=[
        imputer,
        rede_indexer,
        rede_encoder,
        assembler_tree,
        random_forest,
    ]
)

In [0]:
modelo_rf = pipeline_rf.fit(df_treino_ml)

pred_rf_validacao = modelo_rf.transform(df_validacao_ml)

auc_rf = evaluator_auc.evaluate(pred_rf_validacao)

In [0]:
print(f"ROC-AUC validação: {auc_rf:.4f}")

display(
    pred_rf_validacao.groupBy("label", "prediction")
    .count()
    .orderBy("label", "prediction")
)

In [0]:
metricas_rf = calcular_metricas_classificacao(pred_rf_validacao)

for metrica, valor in metricas_rf.items():
    print(f"{metrica}: {valor:.4f}")

print(f"roc_auc: {auc_rf:.4f}")

# 10. Resultados Consolidados

Os modelos da V2 são comparados considerando acurácia, ROC-AUC e desempenho na identificação dos alunos não alfabetizados.

## Resultados V2

In [0]:
resultados_modelos_v2 = [
    (
        "Baseline",
        0.5820,
        None,
        0.0,
        None,
    ),
    (
        "Logistic Regression",
        metricas_logistic["accuracy"],
        auc_logistic,
        metricas_logistic["recall_0"],
        metricas_logistic["f1_0"],
    ),
    (
        "Decision Tree",
        metricas_tree["accuracy"],
        auc_tree,
        metricas_tree["recall_0"],
        metricas_tree["f1_0"],
    ),
    (
        "Random Forest",
        metricas_rf["accuracy"],
        auc_rf,
        metricas_rf["recall_0"],
        metricas_rf["f1_0"],
    ),
]

df_resultados_v2 = spark.createDataFrame(
    resultados_modelos_v2,
    [
        "modelo",
        "accuracy",
        "roc_auc",
        "recall_classe_0",
        "f1_classe_0",
    ],
)

display(df_resultados_v2)

## Comparação V1 × V2

Os resultados da segunda versão são comparados diretamente com a primeira rodada de modelagem.

Como os splits, algoritmos e hiperparâmetros foram mantidos, a diferença de desempenho permite avaliar o impacto do enriquecimento com o IDEB.

In [0]:
resultados_comparacao = [
    (
        "V1",
        "Logistic Regression",
        0.5782,
        0.5686,
        0.0864,
        0.1462,
    ),
    (
        "V2",
        "Logistic Regression",
        metricas_logistic["accuracy"],
        auc_logistic,
        metricas_logistic["recall_0"],
        metricas_logistic["f1_0"],
    ),
    (
        "V1",
        "Decision Tree",
        0.5691,
        0.5359,
        0.4052,
        0.4401,
    ),
    (
        "V2",
        "Decision Tree",
        metricas_tree["accuracy"],
        auc_tree,
        metricas_tree["recall_0"],
        metricas_tree["f1_0"],
    ),
    (
        "V1",
        "Random Forest",
        0.5817,
        0.5769,
        0.1738,
        0.2578,
    ),
    (
        "V2",
        "Random Forest",
        metricas_rf["accuracy"],
        auc_rf,
        metricas_rf["recall_0"],
        metricas_rf["f1_0"],
    ),
]

In [0]:
df_comparacao = spark.createDataFrame(
    resultados_comparacao,
    [
        "versao",
        "modelo",
        "accuracy",
        "roc_auc",
        "recall_classe_0",
        "f1_classe_0",
    ],
)

display(df_comparacao.orderBy("modelo", "versao"))

In [0]:
df_auc_v1 = spark.createDataFrame(
    [
        ("Logistic Regression", 0.5686),
        ("Decision Tree", 0.5359),
        ("Random Forest", 0.5769),
    ],
    ["modelo", "roc_auc_v1"],
)

df_auc_v2 = spark.createDataFrame(
    [
        ("Logistic Regression", auc_logistic),
        ("Decision Tree", auc_tree),
        ("Random Forest", auc_rf),
    ],
    ["modelo", "roc_auc_v2"],
)

In [0]:
display(
    df_auc_v1.join(df_auc_v2, on="modelo")
    .withColumn(
        "ganho_auc",
        F.round(
            F.col("roc_auc_v2") - F.col("roc_auc_v1"),
            4,
        ),
    )
    .orderBy(F.desc("ganho_auc"))
)

# Conclusão da Segunda Rodada de Modelagem

A inclusão das features históricas do IDEB melhorou o desempenho dos três modelos avaliados.

- **Logistic Regression:** ROC-AUC de **0,6064** e acurácia de **59,41%**, apresentando ganho relevante em relação à primeira versão.
- **Decision Tree:** ROC-AUC de **0,5690**, também superior ao resultado obtido na V1.
- **Random Forest:** apresentou o melhor desempenho, com ROC-AUC de **0,6130**, acurácia de **59,90%**, recall de **31,40%** e F1-score de **0,3957** para a classe de alunos não alfabetizados.

Os ganhos de ROC-AUC em relação à V1 foram de **0,0378** na Logistic Regression, **0,0331** na Decision Tree e **0,0361** na Random Forest.

Os resultados indicam que o histórico educacional adicionou sinal preditivo relevante à base. Entretanto, a capacidade discriminatória ainda é moderada, indicando espaço para otimização dos modelos antes da avaliação final no conjunto temporal de 2024.